# Multimodal Embedding Extraction for RSA Analysis

## Zero to Hero: Understanding Each Model

### What is PE-AV?
**PE-AV (Perception Encoder Audio-Visual)** is Meta's multimodal model that learns to align audio, video, and text in a shared embedding space using contrastive learning. Think of it like this:

1. **Input**: Video frames + audio waveform → **Process separately** through video and audio encoders
2. **Fusion**: Combine in a shared transformer backbone
3. **Output**: Normalized embeddings where similar audio-visual content is close in space

**Key insight**: PE-AV has NO special CLS token. It uses **mean pooling** over all tokens from the final transformer layer.

---

### What are we doing with each model?

#### 1. **VideoMAE** (Video-only)
- **What it does**: Self-supervised video understanding by reconstructing masked video patches
- **Architecture**: Vision Transformer (ViT) for videos
- **Our approach**: Extract the **CLS token** from the final layer (VideoMAE has a semantic CLS token unlike PE-AV)
- **Input**: 16 frames sampled uniformly from video
- **Output**: Single 768-d (or 1024-d) normalized vector per video

#### 2. **WavLM** (Audio-only)
- **What it does**: Self-supervised speech representation learning
- **Architecture**: Transformer encoder processing audio at ~50Hz frame rate
- **Our approach**: **Mean pool** over time (all tokens) from final layer, then normalize
- **Input**: 16kHz waveform (2 seconds = 32,000 samples)
- **Output**: Single 768-d (or 1024-d) normalized vector per audio clip

#### 3. **BEATs** (Audio-only)
- **What it does**: Audio understanding via tokenization + masked prediction
- **Architecture**: Similar to WavLM but with discrete audio tokens
- **Our approach**: **Mean pool** over time from final layer, then normalize
- **Input**: 16kHz waveform (2 seconds)
- **Output**: Single 768-d normalized vector per audio clip

#### 4. **PE-AV** (Audio + Video joint)
- **What it does**: Joint audio-visual understanding in shared space
- **Architecture**: Separate audio/video encoders → shared transformer → mean pooling
- **Our approach**: **Mean pool** over all tokens from final shared layer, then normalize
- **Input**: Video frames + audio waveform
- **Output**: Single normalized vector in joint A-V space

---

### Why Normalize?
**RSA (Representational Similarity Analysis)** compares *similarity structures* between models.
- We compute pairwise distances/similarities between all stimuli
- Normalization ensures we measure **angular distance** (cosine similarity) not magnitude
- PE-AV outputs are **already normalized** internally, so we must normalize all others!

---

### Critical Compatibility Requirements
1. ✅ **All embeddings must be L2-normalized**
2. ✅ **Use mean pooling for models without semantic CLS** (WavLM, BEATs, PE-AV)
3. ✅ **Match temporal alignment** (2-second clips for audio, 16 frames for video)
4. ✅ **Same distance metric** (1 - cosine similarity or Euclidean on normalized vectors)


In [ ]:
import torch
import torchaudio
import torchvision
import numpy as np
from pathlib import Path
from sklearn.metrics import pairwise_distances
import warnings
warnings.filterwarnings("ignore")


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if DEVICE.type == "cuda" else torch.float32


print("Device:", DEVICE)
print("Data type:", DTYPE)

In [ ]:
def l2_normalize(x: torch.Tensor):
    """
    L2 normalization: scale vectors to unit length.
    This ensures cosine similarity = dot product.
    
    Args:
        x: Tensor of shape (batch, dim)
    
    Returns:
        Normalized tensor of same shape
    """
    return x / (x.norm(dim=-1, keepdim=True) + 1e-8)

## 1. VideoMAE Embedding Extraction

### Why CLS token for VideoMAE?
Unlike PE-AV, VideoMAE follows the **ViT (Vision Transformer)** paradigm:
- A special `[CLS]` token is prepended to the sequence
- This token is trained to aggregate global video semantics
- The CLS token at the **final layer** is the standard readout for downstream tasks

### Frame Sampling
We sample **16 frames uniformly** across the video duration, matching PE-AV's approach.

In [ ]:
from transformers import VideoMAEImageProcessor, VideoMAEModel
from decord import VideoReader, cpu
import decord

decord.bridge.set_bridge("torch")

VIDEOMAE_PATH = Path("/path/to/local/videomae")

processor_v = VideoMAEImageProcessor.from_pretrained(VIDEOMAE_PATH)
model_v = VideoMAEModel.from_pretrained(VIDEOMAE_PATH).to(DEVICE).eval()


def load_video_fixed(path, num_frames=16):
    """
    Load exactly num_frames uniformly sampled from a video.
    
    Args:
        path: Path to video file
        num_frames: Number of frames to extract (default: 16)
    
    Returns:
        Tensor of shape (num_frames, C, H, W)
    """
    vr = VideoReader(str(path), ctx=cpu(0))
    idx = np.linspace(0, len(vr) - 1, num_frames).astype(np.int64)
    frames = vr.get_batch(idx)  # torch Tensor via decord bridge
    return frames.permute(0, 3, 1, 2)  # (T, C, H, W)


def extract_videomae(video_paths):
    """
    Extract VideoMAE embeddings using CLS token from final layer.
    
    Architecture:
        Video → ViT encoder → [CLS, patch_1, ..., patch_N] → CLS token → L2 norm
    
    Args:
        video_paths: List of video file paths
    
    Returns:
        numpy array of shape (len(video_paths), embed_dim)
    """
    embs = []
    with torch.inference_mode():
        for vp in video_paths:
            frames = load_video_fixed(vp, num_frames=16)
            
            # Processor handles normalization and resizing
            inputs = processor_v(
                list(frames), return_tensors="pt"
            ).to(DEVICE)

            out = model_v(**inputs)
            
            # Extract CLS token (position 0) from final layer
            cls = out.last_hidden_state[:, 0]  # (1, D)
            
            # L2 normalize for RSA compatibility
            cls = l2_normalize(cls.float())
            embs.append(cls.cpu())

    return torch.cat(embs).numpy()


## 2. WavLM Embedding Extraction

### Why Mean Pooling for WavLM?
WavLM is a **Wav2Vec 2.0** variant:
- No special CLS token (unlike BERT/ViT)
- Outputs a sequence of frame-level representations (~50 Hz)
- Standard practice: **mean pool** over time to get utterance-level embedding

This matches PE-AV's approach of averaging all tokens.

### Temporal Alignment
We fix audio to **exactly 2 seconds** (32,000 samples @ 16kHz) to match your video bins.

In [ ]:
from transformers import WavLMModel, Wav2Vec2FeatureExtractor

WAVLM_PATH = Path("/path/to/local/wavlm")

processor_a = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_PATH)
model_a = WavLMModel.from_pretrained(WAVLM_PATH).to(DEVICE).eval()


def load_audio_fixed(path, target_len, sr=16000):
    """
    Load audio and fix length to exactly target_len samples.
    
    Args:
        path: Audio file path
        target_len: Desired length in samples (e.g., 32000 for 2s @ 16kHz)
        sr: Target sample rate
    
    Returns:
        Mono audio tensor of length target_len
    """
    wav, fs = torchaudio.load(path)
    
    # Resample if needed
    if fs != sr:
        wav = torchaudio.functional.resample(wav, fs, sr)
    
    # Convert to mono
    wav = wav.mean(0)
    
    # Pad or truncate to exact length
    if wav.numel() < target_len:
        wav = torch.nn.functional.pad(wav, (0, target_len - wav.numel()))
    else:
        wav = wav[:target_len]
    
    return wav


def extract_wavlm(audio_paths, T=2.0, sr=16000):
    """
    Extract WavLM embeddings using mean pooling over time.
    
    Architecture:
        Audio waveform → CNN feature extractor → Transformer encoder → 
        Mean pool over time → L2 norm
    
    Args:
        audio_paths: List of audio file paths
        T: Duration in seconds (default: 2.0)
        sr: Sample rate (default: 16000)
    
    Returns:
        numpy array of shape (len(audio_paths), embed_dim)
    """
    target_len = int(T * sr)
    embs = []

    with torch.inference_mode():
        for ap in audio_paths:
            wav = load_audio_fixed(ap, target_len, sr)
            
            # Feature extractor normalizes to zero mean
            inp = processor_a(
                wav.numpy(),
                sampling_rate=sr,
                return_tensors="pt"
            )

            inp = {k: v.to(DEVICE) for k, v in inp.items()}
            out = model_a(**inp)

            # Mean pool over time dimension (dim=1)
            # Shape: (1, T', D) → (1, D)
            h = out.last_hidden_state.mean(dim=1)
            
            # L2 normalize
            h = l2_normalize(h.float())
            embs.append(h.cpu())

    return torch.cat(embs).numpy()

## 3. BEATs Embedding Extraction

### BEATs Architecture
BEATs (Bidirectional Encoder representation from Audio Transformers):
- Tokenizes audio into discrete units
- Uses masked prediction (like BERT for text)
- Also has **no CLS token** → use mean pooling

Same temporal alignment and normalization as WavLM.

In [ ]:
from transformers import AutoModel

BEATS_PATH = Path("/path/to/local/beats")
model_b = AutoModel.from_pretrained(BEATS_PATH).to(DEVICE).eval()


def extract_beats(audio_paths, T=2.0, sr=16000):
    """
    Extract BEATs embeddings using mean pooling over time.
    
    Architecture:
        Audio waveform → Tokenizer → Transformer encoder → 
        Mean pool over time → L2 norm
    
    Args:
        audio_paths: List of audio file paths
        T: Duration in seconds (default: 2.0)
        sr: Sample rate (default: 16000)
    
    Returns:
        numpy array of shape (len(audio_paths), embed_dim)
    """
    target_len = int(T * sr)
    embs = []

    with torch.inference_mode():
        for ap in audio_paths:
            wav = load_audio_fixed(ap, target_len, sr)
            wav = wav.unsqueeze(0).to(DEVICE)  # (1, T)

            out = model_b(wav)

            # BEATs may use different output attribute names
            # Handle both last_hidden_state and extract_features
            if hasattr(out, "last_hidden_state"):
                h = out.last_hidden_state
            else:
                h = out.extract_features

            # Mean pool over time dimension
            h = h.mean(dim=1)
            
            # L2 normalize
            h = l2_normalize(h.float())
            embs.append(h.cpu())

    return torch.cat(embs).numpy()

## 4. PE-AV Embedding Extraction

### PE-AV Architecture Deep Dive

```
Video frames → Video Encoder (ViT-like) ──┐
                                           ├─→ Shared Transformer Backbone → Mean Pool → L2 Norm
Audio waveform → Audio Encoder (CNN+Trans)─┘
```

**Key differences from other models:**
1. **Dual encoders**: Separate preprocessing for audio and video
2. **Shared backbone**: Features are fused in late transformer layers
3. **Mean pooling**: No semantic CLS token, averages ALL tokens
4. **Contrastive training**: Embeddings are already optimized for similarity matching

### Critical Fix: You Must Normalize PE-AV Embeddings!

You mentioned you extracted PE-AV embeddings but **didn't normalize them**. This is the root issue!

**Why this matters for RSA:**
- RSA compares **similarity structures** (correlation of distance matrices)
- Unnormalized embeddings include magnitude information
- Other models are normalized → incomparable distance scales

**Solution:** Re-normalize your existing PE-AV embeddings:
```python
pe_av_normalized = pe_av_embeddings / (np.linalg.norm(pe_av_embeddings, axis=1, keepdims=True) + 1e-8)
```

Or extract fresh embeddings using the code below.

In [ ]:
from transformers import PeAudioVideoModel, PeAudioVideoProcessor

PEAV_PATH = "facebook/pe-av-large"  # or your local path

processor = PeAudioVideoProcessor.from_pretrained(PEAV_PATH)
model = PeAudioVideoModel.from_pretrained(PEAV_PATH).to(DEVICE).eval()


def extract_pe_av(video_paths, audio_paths, batch_size=4):
    """
    Extract PE-AV joint audio-visual embeddings.
    
    Architecture:
        Video → Video Encoder → ┐
                                 ├─→ Shared Transformer → Mean Pool → L2 Norm
        Audio → Audio Encoder → ┘
    
    CRITICAL: PE-AV has NO semantic CLS token. The model:
    1. Processes audio and video through separate encoders
    2. Fuses in a shared transformer backbone
    3. Mean pools ALL tokens from the final shared layer
    4. Returns normalized embeddings (but we re-normalize for safety)
    
    Args:
        video_paths: List of video file paths
        audio_paths: List of audio file paths (must match video_paths length)
        batch_size: Batch size for processing
    
    Returns:
        numpy array of shape (len(video_paths), embed_dim)
    """
    assert len(video_paths) == len(audio_paths), "Video and audio lists must match!"
    
    out = []

    with torch.inference_mode():
        for i in range(0, len(video_paths), batch_size):
            v = video_paths[i:i + batch_size]
            a = audio_paths[i:i + batch_size]

            # Processor handles:
            # - Video: frame extraction, resizing, normalization
            # - Audio: resampling to 16kHz, feature extraction
            inputs = processor(
                videos=v,
                audio=a,
                return_tensors="pt",
                padding=True
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

            # Use mixed precision for efficiency
            with torch.autocast(device_type=DEVICE.type, dtype=DTYPE):
                outputs = model(
                    **inputs,
                    output_hidden_states=True
                )

            # Method 1: Use the direct audio_video_embeds output
            # This is ALREADY mean-pooled and normalized internally
            emb = outputs.audio_video_embeds  # (B, D)
            
            # Re-normalize to ensure L2 norm = 1 (should already be, but safety first)
            emb = l2_normalize(emb.float())
            
            out.append(emb.cpu())

    return torch.cat(out).numpy()


def extract_pe_av_manual_pooling(video_paths, audio_paths, batch_size=4):
    """
    Alternative: Manual mean pooling from hidden states.
    This explicitly shows what PE-AV does internally.
    
    Use this if you want to exactly replicate the pooling strategy.
    """
    assert len(video_paths) == len(audio_paths)
    
    out = []

    with torch.inference_mode():
        for i in range(0, len(video_paths), batch_size):
            v = video_paths[i:i + batch_size]
            a = audio_paths[i:i + batch_size]

            inputs = processor(
                videos=v,
                audio=a,
                return_tensors="pt",
                padding=True
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

            with torch.autocast(device_type=DEVICE.type, dtype=DTYPE):
                outputs = model(
                    **inputs,
                    output_hidden_states=True
                )

            # Extract final shared backbone layer
            # Shape: [B, N_tokens, D]
            h = outputs.hidden_states[-1]

            # Mean pooling over tokens (PE-AV has no semantic CLS)
            # Shape: [B, N_tokens, D] → [B, D]
            h = h.mean(dim=1)

            # L2 normalize for RSA compatibility
            h = l2_normalize(h.float())

            out.append(h.cpu())

    return torch.cat(out).numpy()


# Use extract_pe_av for most cases (cleaner)
# Use extract_pe_av_manual_pooling if you want explicit control

## 5. RSA Preparation

### What is RSA?
**Representational Similarity Analysis** compares how different models represent the same stimuli.

**Steps:**
1. Extract embeddings for all stimuli (videos/audio clips)
2. Compute pairwise distances → **Representational Dissimilarity Matrix (RDM)**
3. Compare RDMs between models using correlation (Spearman/Pearson)

**Key insight:** We're comparing *patterns of similarity*, not raw embeddings.

### Distance Metrics for Normalized Embeddings
For L2-normalized vectors, these are equivalent:
- Euclidean distance
- 1 - Cosine similarity
- Angular distance

We'll use **cosine distance** (1 - cosine similarity) for clarity.

In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr, pearsonr


def compute_rdm(embeddings, metric='cosine'):
    """
    Compute Representational Dissimilarity Matrix.
    
    Args:
        embeddings: numpy array of shape (n_stimuli, embed_dim)
        metric: Distance metric ('cosine', 'euclidean', 'correlation')
    
    Returns:
        RDM: Square matrix of shape (n_stimuli, n_stimuli)
    """
    # Verify normalization (should be close to 1.0)
    norms = np.linalg.norm(embeddings, axis=1)
    if not np.allclose(norms, 1.0, atol=1e-3):
        print(f"Warning: Embeddings not normalized! Norms range: [{norms.min():.3f}, {norms.max():.3f}]")
    
    # Compute pairwise distances
    distances = pdist(embeddings, metric=metric)
    rdm = squareform(distances)
    
    return rdm


def compare_rdms(rdm1, rdm2, method='spearman'):
    """
    Compare two RDMs using correlation.
    
    Args:
        rdm1, rdm2: Square RDMs of same shape
        method: 'spearman' or 'pearson'
    
    Returns:
        correlation: Correlation coefficient
        p_value: Statistical significance
    """
    # Extract upper triangle (excluding diagonal)
    mask = np.triu(np.ones_like(rdm1, dtype=bool), k=1)
    vec1 = rdm1[mask]
    vec2 = rdm2[mask]
    
    if method == 'spearman':
        corr, pval = spearmanr(vec1, vec2)
    else:
        corr, pval = pearsonr(vec1, vec2)
    
    return corr, pval


def normalize_existing_embeddings(embeddings):
    """
    Normalize existing (unnormalized) embeddings.
    Use this to fix your PE-AV embeddings!
    
    Args:
        embeddings: numpy array of shape (n_stimuli, embed_dim)
    
    Returns:
        Normalized embeddings
    """
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / (norms + 1e-8)

## 6. Example Usage

### Full Pipeline Example

In [ ]:
# Example file lists (replace with your data)
video_files = [
    "/path/to/video_clip_001.mp4",
    "/path/to/video_clip_002.mp4",
    # ... more videos
]

audio_files = [
    "/path/to/audio_clip_001.wav",
    "/path/to/audio_clip_002.wav",
    # ... more audio (must match video order)
]

# Extract embeddings from each model
print("Extracting VideoMAE embeddings...")
videomae_embs = extract_videomae(video_files)

print("Extracting WavLM embeddings...")
wavlm_embs = extract_wavlm(audio_files, T=2.0)

print("Extracting BEATs embeddings...")
beats_embs = extract_beats(audio_files, T=2.0)

print("Extracting PE-AV embeddings...")
peav_embs = extract_pe_av(video_files, audio_files)

# Compute RDMs
rdm_videomae = compute_rdm(videomae_embs, metric='cosine')
rdm_wavlm = compute_rdm(wavlm_embs, metric='cosine')
rdm_beats = compute_rdm(beats_embs, metric='cosine')
rdm_peav = compute_rdm(peav_embs, metric='cosine')

# Compare models
print("\n=== RSA Results ===")

corr, pval = compare_rdms(rdm_peav, rdm_videomae)
print(f"PE-AV vs VideoMAE: r={corr:.3f}, p={pval:.2e}")

corr, pval = compare_rdms(rdm_peav, rdm_wavlm)
print(f"PE-AV vs WavLM: r={corr:.3f}, p={pval:.2e}")

corr, pval = compare_rdms(rdm_peav, rdm_beats)
print(f"PE-AV vs BEATs: r={corr:.3f}, p={pval:.2e}")

corr, pval = compare_rdms(rdm_wavlm, rdm_beats)
print(f"WavLM vs BEATs: r={corr:.3f}, p={pval:.2e}")

## 7. Fix Your Existing PE-AV Embeddings

If you already extracted PE-AV embeddings but didn't normalize them, use this:

In [ ]:
# Load your existing PE-AV embeddings
# peav_unnormalized = np.load("/path/to/your/peav_embeddings.npy")

# Normalize them
# peav_normalized = normalize_existing_embeddings(peav_unnormalized)

# Verify normalization
# norms = np.linalg.norm(peav_normalized, axis=1)
# print(f"Norm range: [{norms.min():.6f}, {norms.max():.6f}]")  # Should be ~1.0

# Save normalized version
# np.save("/path/to/your/peav_embeddings_normalized.npy", peav_normalized)

## Summary: Key Takeaways

### ✅ Critical Fixes Applied:

1. **Normalization everywhere**: All embeddings are L2-normalized
2. **Correct pooling strategies**:
   - VideoMAE: CLS token (has semantic CLS)
   - WavLM, BEATs, PE-AV: Mean pooling (no semantic CLS)
3. **Temporal alignment**: 2-second audio clips, 16 frames for video
4. **Proper PE-AV usage**: Using `audio_video_embeds` output or manual mean pooling

### 🎯 RSA Compatibility:

- All models now produce comparable embeddings
- Distance metrics are on the same scale (cosine distance)
- RDM comparison is statistically valid

### 🔧 For Your Existing Data:

**If you already have PE-AV embeddings (unnormalized):**
```python
peav_normalized = normalize_existing_embeddings(peav_embeddings)
```

**Then proceed with RSA as normal.**

### 📚 References:

- PE-AV: [HuggingFace Model Card](https://huggingface.co/facebook/pe-av-large)
- VideoMAE: [Paper](https://arxiv.org/abs/2203.12602)
- WavLM: [Paper](https://arxiv.org/abs/2110.13900)
- BEATs: [Paper](https://arxiv.org/abs/2212.09058)
- RSA: [Kriegeskorte et al., 2008](https://doi.org/10.3389/neuro.06.004.2008)
